# Backtest — Table 2 Replication (Krauss et al. 2017)

Computes all metrics from Table 2 for **DNN, GBT, ENS1, MKT** at k=10.

Outputs:
- `data/perf_paper_period.csv`  — 1992-12-17 to 2015-10-15 (5,750 days)
- `data/perf_post_paper.csv`   — 2015-10-16 to 2024-09-25 (2,249 days)

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

DATA  = '../data'
TC    = 0.002          # 20 bps round-trip per day
TDPY  = 250

PAPER_START, PAPER_END = '1992-12-17', '2015-10-15'
EXT_START,   EXT_END   = '2015-10-16', '2024-09-25'

## 1. Load all data

In [ ]:
# ── Portfolio returns ─────────────────────────────────────────────────────────
dnn_port = pd.read_csv(f'{DATA}/dnn_backtest_results_k10.csv', parse_dates=['date'])

gbt_raw  = pd.read_csv(f'{DATA}/gbt_daily_returns.csv', parse_dates=['date'])
gbt_port = gbt_raw.rename(columns={'raw_return': 'raw_ret', 'net_return': 'net_ret'})

ens_all  = pd.read_csv(f'{DATA}/ensemble_portfolio_returns.csv', parse_dates=['date'])
ens1_port = (ens_all[(ens_all['ensemble']=='ENS1') & (ens_all['k']==10)]
             .rename(columns={'portfolio_ret': 'raw_ret'})
             .assign(net_ret=lambda d: d['raw_ret'] - TC)
             .reset_index(drop=True))

mkt = pd.read_csv(f'{DATA}/sp500_returns.csv', parse_dates=['Date'])
mkt = mkt.rename(columns={'Date': 'date', '^GSPC': 'raw_ret'})

# ── Predictions (for PT test) ─────────────────────────────────────────────────
feat      = pd.read_csv(f'{DATA}/features.csv', parse_dates=['date'])[['permno','date','Y']]

dnn_pred  = pd.read_csv(f'{DATA}/dnn_predictions.csv', parse_dates=['date'])
dnn_pred  = dnn_pred.merge(feat, on=['permno','date'], how='left')

gbt_pred  = pd.read_csv(f'{DATA}/gbt_predictions.csv', parse_dates=['date'])
gbt_pred  = gbt_pred.merge(feat, on=['permno','date'], how='left')

ens_pred  = pd.read_csv(f'{DATA}/ensemble_predictions.csv', parse_dates=['date'])
ens_pred  = ens_pred.merge(feat, on=['permno','date'], how='left')

print('DNN port:', dnn_port.shape, '| GBT port:', gbt_port.shape,
      '| ENS1 port:', ens1_port.shape, '| MKT:', mkt.shape)

## 2. Metric functions

In [ ]:
def nw_se(series, lags=1):
    """Newey-West SE (1-lag), matching paper footnote convention."""
    y = series.values
    X = np.ones((len(y), 1))
    res = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': lags}, use_t=False)
    return float(res.bse[0])


def pt_test(y_true, y_score):
    """Pesaran-Timmermann (1992) test on stock-level directional accuracy."""
    f  = (y_score > 0.5).astype(float)
    y  = y_true.astype(float)
    N  = len(y)
    P1 = y.mean()
    P2 = f.mean()
    Phat  = (f == y).mean()
    Pstar = P1*P2 + (1-P1)*(1-P2)
    var_Phat  = Pstar*(1-Pstar)/N
    var_Pstar = ((2*P2-1)**2*P1*(1-P1) + (2*P1-1)**2*P2*(1-P2)) / N
    denom = var_Phat - var_Pstar
    return float((Phat - Pstar) / np.sqrt(denom)) if denom > 0 else np.nan


def compute_metrics(ret, long_ret, short_ret, pt_stat=np.nan, is_market=False):
    """All Table-2 metrics for one return series."""
    n      = len(ret)
    mu     = ret.mean()
    sigma  = ret.std()
    total  = (1 + ret).prod() - 1
    ann_r  = (1 + total)**(TDPY/n) - 1
    ann_v  = sigma * np.sqrt(TDPY)
    se     = nw_se(ret)
    t_nw   = mu / se
    cum    = (1 + ret).cumprod()
    dd     = (cum - cum.cummax()) / cum.cummax()
    max_dd = abs(dd.min())
    calmar = ann_r / max_dd if max_dd > 0 else np.nan
    var1   = np.percentile(ret, 1)
    cvar1  = ret[ret <= var1].mean()
    var5   = np.percentile(ret, 5)
    cvar5  = ret[ret <= var5].mean()

    d = {
        'Mean return (long)':        round(long_ret.mean(), 4) if not is_market else '-',
        'Mean return (short)':       round(short_ret.mean(), 4) if not is_market else '-',
        'Mean return':               round(mu, 4),
        'Standard error (NW)':       round(se, 4),
        't-statistic (NW)':          round(t_nw, 4),
        'PT test statistic':         round(pt_stat, 4) if (not np.isnan(pt_stat) and not is_market) else '-',
        'Minimum':                   round(ret.min(), 4),
        'Quartile 1':                round(np.percentile(ret, 25), 4),
        'Median':                    round(np.median(ret), 4),
        'Quartile 3':                round(np.percentile(ret, 75), 4),
        'Maximum':                   round(ret.max(), 4),
        'Standard deviation':        round(sigma, 4),
        'Skewness':                  round(stats.skew(ret), 4),
        'Kurtosis':                  round(stats.kurtosis(ret), 4),
        'Historical 1-percent VaR':  round(var1, 4),
        'Historical 1-percent CVaR': round(cvar1, 4),
        'Historical 5-percent VaR':  round(var5, 4),
        'Historical 5-percent CVaR': round(cvar5, 4),
        'Maximum drawdown':          round(max_dd, 4),
        'Calmar ratio':              round(calmar, 4),
        'Share with return > 0':     round((ret > 0).mean(), 4),
    }
    return d

## 3. Helper — slice by period and build table

In [ ]:
def slice_port(df, start, end):
    return df[(df['date'] >= start) & (df['date'] <= end)].reset_index(drop=True)

def slice_pred(df, start, end):
    return df[(df['date'] >= start) & (df['date'] <= end)].reset_index(drop=True)


def build_period_table(start, end):
    """
    Returns a DataFrame with rows = metrics, columns =
    [metric, DNN_before, DNN_after, GBT_before, GBT_after,
             ENS1_before, ENS1_after, MKT]
    """
    # ── DNN ──────────────────────────────────────────────────────────────────
    d = slice_port(dnn_port, start, end)
    dp = slice_pred(dnn_pred, start, end)
    pt_dnn = pt_test(dp['Y'].values, dp['prob_outperform'].values)
    zero = pd.Series(np.zeros(len(d)))   # placeholder for after-TC long/short
    m_dnn_raw = compute_metrics(d['raw_ret'], d['long_ret'], d['short_ret'], pt_dnn)
    m_dnn_net = compute_metrics(d['net_ret'], d['long_ret'], d['short_ret'], np.nan)

    # ── GBT ──────────────────────────────────────────────────────────────────
    g = slice_port(gbt_port, start, end)
    gp = slice_pred(gbt_pred, start, end)
    pt_gbt = pt_test(gp['Y'].values, gp['prob_GBT'].values)
    m_gbt_raw = compute_metrics(g['raw_ret'], g['long_ret'], g['short_ret'], pt_gbt)
    m_gbt_net = compute_metrics(g['net_ret'], g['long_ret'], g['short_ret'], np.nan)

    # ── ENS1 ─────────────────────────────────────────────────────────────────
    e = slice_port(ens1_port, start, end)
    ep = slice_pred(ens_pred, start, end)
    pt_ens = pt_test(ep['Y'].values, ep['ENS1'].values)
    m_ens_raw = compute_metrics(e['raw_ret'], e['long_ret'], e['short_ret'], pt_ens)
    m_ens_net = compute_metrics(e['net_ret'], e['long_ret'], e['short_ret'], np.nan)

    # ── Market ────────────────────────────────────────────────────────────────
    mk = slice_port(mkt, start, end)
    dummy = pd.Series(np.zeros(len(mk)))
    m_mkt = compute_metrics(mk['raw_ret'], dummy, dummy, is_market=True)

    # ── Assemble ──────────────────────────────────────────────────────────────
    metrics = list(m_dnn_raw.keys())
    table = pd.DataFrame({
        'metric':      metrics,
        'DNN_before_TC':  [m_dnn_raw[k] for k in metrics],
        'DNN_after_TC':   [m_dnn_net[k] for k in metrics],
        'GBT_before_TC':  [m_gbt_raw[k] for k in metrics],
        'GBT_after_TC':   [m_gbt_net[k] for k in metrics],
        'ENS1_before_TC': [m_ens_raw[k] for k in metrics],
        'ENS1_after_TC':  [m_ens_net[k] for k in metrics],
        'MKT':            [m_mkt[k] for k in metrics],
    })
    return table

## 4. Compute — Paper period (1992-12-17 to 2015-10-15)

In [ ]:
paper_tbl = build_period_table(PAPER_START, PAPER_END)
paper_tbl

## 5. Compute — Extension period (2015-10-16 to 2024-09-25)

In [ ]:
ext_tbl = build_period_table(EXT_START, EXT_END)
ext_tbl

## 6. Save

In [ ]:
paper_tbl.to_csv(f'{DATA}/perf_paper_period.csv', index=False)
ext_tbl.to_csv(  f'{DATA}/perf_post_paper.csv',   index=False)

print(f'Saved perf_paper_period.csv  ({len(paper_tbl)} metrics)')
print(f'Saved perf_post_paper.csv    ({len(ext_tbl)} metrics)')